In [2]:
import numpy as np
from PIL import Image

def combine_images_square(upper_path, lower_paths, output_path):
    # 加载所有图片
    upper = Image.open(upper_path).convert('RGB')
    lowers = [Image.open(p).convert('RGB') for p in lower_paths]

    # 阶段1：处理下方三图横向排列
    # ----------------------------------------
    # 统一高度（取最小高度避免拉伸）
    lower_heights = [img.height for img in lowers]
    base_height = min(lower_heights)
    
    # 保持宽高比缩放
    lower_resized = []
    total_lower_width = 0
    for img in lowers:
        scale_ratio = base_height / img.height
        new_width = int(img.width * scale_ratio)
        resized = img.resize((new_width, base_height), Image.LANCZOS)
        lower_resized.append(resized)
        total_lower_width += new_width

    # 横向拼接
    lower_combined = Image.new('RGB', (total_lower_width, base_height))
    x_offset = 0
    for img in lower_resized:
        lower_combined.paste(img, (x_offset, 0))
        x_offset += img.width

    # 阶段2：处理上方图与下方组合图的纵向排列
    # ----------------------------------------
    # 缩放上方图宽度与下方组合图一致
    upper_width = lower_combined.width
    upper_height = int(upper.height * (upper_width / upper.width))
    upper_resized = upper.resize((upper_width, upper_height), Image.LANCZOS)

    # 计算总尺寸
    total_height = upper_height + base_height
    final_size = max(upper_width, total_height)

    # 阶段3：构建最终正方形
    # ----------------------------------------
    final_img = Image.new('RGB', (final_size, final_size), (255, 255, 255))
    # 上方图居中放置
    final_img.paste(upper_resized, ((final_size - upper_width)//2, 0))
    # 下方组合图居中放置
    final_img.paste(lower_combined, ((final_size - total_lower_width)//2, upper_height))

    # 等比缩放（当需要严格正方形时启用）
    if final_size != upper_width:
        final_img = final_img.resize((final_size, final_size), Image.LANCZOS)

    final_img.save(output_path)
    print(f"已保存至 {output_path}，尺寸：{final_img.size}")

# 使用示例
# combine_images_square(
#     upper_path="fig1_challenge_subfig11.png",
#     lower_paths=["1d_mit.png", "2d_mit.png", "3d_mit.png"],
#     output_path="combined_square.png"
# )
combine_images_square(
    upper_path="fig1_challenge_subfig11.png",
    lower_paths=["3d_tongji.png", "2d_tongji.png", "1d_tongji.png"],
    output_path="combined_square.png"
)

已保存至 combined_square.png，尺寸：(4248, 4248)
